# Simulated pure-space spectral diagnostic: true smooth=0.5

This notebook borrows the real GEMS hourly grid/missingness pattern, but replaces the TCO values with simulated independent spatial Matérn fields.

The simulation truth is:

- `smooth=0.5`
- isotropic range / variance / nugget fixed in the simulation block below
- mild linear mean trend, removed by `mean_design='latlon'`

It then fits the same isotropic ClusterB2 Vecchia pure-space model across resolution thinning levels and draws the same diagnostics as the real-data notebooks:

- parameter trajectories as a 3x3 matrix
- low-frequency data spectrum vs full theoretical spectrum as a 3x4 matrix


In [ ]:
import gc
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.special import gamma as scipy_gamma
from scipy.special import kv as scipy_kv

LOCAL_SRC = '/Users/joonwonlee/Documents/GEMS_TCO-1/src'
AMAREL_SRC = '/home/jl2815/tco'
SRC = AMAREL_SRC if Path(AMAREL_SRC).exists() else LOCAL_SRC
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed
from GEMS_TCO.kernels_space_iso_cluster_052426 import (
    ClusterSpaceIsoTrendVecchiaFit,
    ClusterSpaceIsoNoNuggetTrendVecchiaFit,
)

DEVICE = torch.device('cpu')
DTYPE = torch.float64
ROUND_DECIMALS = 4

pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 200)
print('SRC:', SRC)
print('device:', DEVICE)


In [ ]:
# Experiment config
YEAR = '2024'
MONTH = 7
DAY_IDX = 2                 # 0-based: 2 -> 2024-07-03
HOUR_IDX_LIST = list(range(8))
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]

FIT_SMOOTHS = [0.3, 0.5]
SMOOTH = FIT_SMOOTHS[0]  # compatibility default; actual fits loop over FIT_SMOOTHS
RESOLUTION_STRIDES = [8, 4, 2, 1]
MEAN_DESIGN = 'latlon'

TRUE_SMOOTH = 0.5
TRUE_PARAMS = {'sigmasq': 12.0, 'range': 0.28, 'nugget': 2.0}
SIM_MEAN = {'intercept': 300.0, 'lat_slope': 1.5, 'lon_slope': -0.8}
SIM_SEED = 20260508

CLUSTER_SPEC = {
    'block_shape': (4, 4),
    'n_neighbor_blocks': 2,
    'target_chunk_size': 128,
    'min_target_points': 1,
}

VARIANTS = {
    'nugget_free': {
        'class': ClusterSpaceIsoTrendVecchiaFit,
        'model': 'ClusterSpaceIso_4x4_B2_exactloc',
        'kernel': 'cluster_space_iso_4x4_b2_exactloc',
        'p_labels': ['sigmasq', 'range', 'nugget'],
        'init': {'sigmasq': 13.0, 'range': 0.25, 'nugget': 2.5},
    },
    'nugget0': {
        'class': ClusterSpaceIsoNoNuggetTrendVecchiaFit,
        'model': 'ClusterSpaceIsoNoNugget_4x4_B2_exactloc',
        'kernel': 'cluster_space_iso_nugget0_4x4_b2_exactloc',
        'p_labels': ['sigmasq', 'range'],
        'init': {'sigmasq': 13.0, 'range': 0.25},
    },
}

LBFGS_LR = 1.0
LBFGS_STEPS_FULL = 8
LBFGS_EVAL = 20
LBFGS_HIST = 10
GRAD_TOL = 1e-5

RUN_FULL = True
RUN_SPECTRUM = True

PURE_SPACE_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/pure_space')
OUT_DIR = PURE_SPACE_DIR / 'log'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PREFIX = 'sim_pure_space_spectral_s05_clusterb2_2dthin_052426'

print('day:', f'{YEAR}{MONTH:02d}{DAY_IDX + 1:02d}')
print('hours:', HOUR_IDX_LIST)
print('strides:', RESOLUTION_STRIDES)
print('true smooth:', TRUE_SMOOTH)
print('fit smooths:', FIT_SMOOTHS)
print('variants:', list(VARIANTS))


def smooth_tag(x):
    return str(float(x)).replace('.', 'p')

In [ ]:
def phys_to_log(init, p_labels):
    return [np.log(init[p]) for p in p_labels]


def backmap(raw, p_labels, variant):
    est = {p: float(np.exp(raw[i])) for i, p in enumerate(p_labels)}
    if 'nugget' not in est:
        est['nugget'] = 0.0
    return est


def make_full_params(variant):
    spec = VARIANTS[variant]
    return [torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True) for v in phys_to_log(spec['init'], spec['p_labels'])]


def count_valid(input_map):
    return int(sum(((~torch.isnan(v[:, 2])) & (~torch.isnan(v[:, 0])) & (~torch.isnan(v[:, 1]))).sum().item() for v in input_map.values()))


def space_diag(model):
    if hasattr(model, 'cluster_summary'):
        return dict(model.cluster_summary())

    groups = getattr(model, 'Batched_Groups', []) or []
    if not groups:
        return {'n_batches': 0, 'n_tails': 0, 'mean_m': 0.0, 'max_m': 0, 'largest_batch_n': 0}
    ns = np.asarray([int(g['target_idx'].shape[0]) for g in groups], dtype=np.int64)
    ms = np.asarray([int(g['max_m']) for g in groups], dtype=np.int64)
    return {
        'n_batches': int(len(groups)),
        'n_tails': int(ns.sum()),
        'mean_m': float(ms.mean()),
        'max_m': int(ms.max()),
        'largest_batch_n': int(ns.max()),
    }


def round_df(df, digits=ROUND_DECIMALS):
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(digits)
    return out


def empty_cache():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


def resolution_label(stride):
    return f'x{int(stride)}'


In [ ]:
loader = load_data_dynamic_processed(config.mac_data_load_path)
df_map, _, _, monthly_mean = loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=[1, 1],
    mm_cond_number=10,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=True,
)
month_keys = sorted(df_map.keys())
day_keys = month_keys[DAY_IDX * 8:(DAY_IDX + 1) * 8]
if len(day_keys) != 8:
    raise ValueError(f'Day {DAY_IDX} has {len(day_keys)} slices, expected 8')

first_df = df_map[day_keys[0]].reset_index(drop=True)
grid_order = (
    first_df
    .assign(_orig_idx=np.arange(len(first_df)))
    .sort_values(['Longitude', 'Latitude', '_orig_idx'], kind='mergesort')['_orig_idx']
    .to_numpy(dtype=np.int64)
)
grid_coords_full = first_df.iloc[grid_order][['Latitude', 'Longitude']].to_numpy(dtype=np.float64)

_lat_key = np.round(grid_coords_full[:, 0], 10)
_lon_key = np.round(grid_coords_full[:, 1], 10)
sim_lat_vals = np.sort(np.unique(_lat_key))
sim_lon_vals = np.sort(np.unique(_lon_key))
sim_lat_to_row = {float(v): i for i, v in enumerate(sim_lat_vals)}
sim_lon_to_col = {float(v): i for i, v in enumerate(sim_lon_vals)}
sim_local_to_row = np.asarray([sim_lat_to_row[float(v)] for v in _lat_key], dtype=np.int64)
sim_local_to_col = np.asarray([sim_lon_to_col[float(v)] for v in _lon_key], dtype=np.int64)
SIM_N_LAT, SIM_N_LON = len(sim_lat_vals), len(sim_lon_vals)
SIM_LAT_STEP = float(np.median(np.diff(sim_lat_vals))) if SIM_N_LAT > 1 else 1.0
SIM_LON_STEP = float(np.median(np.diff(sim_lon_vals))) if SIM_N_LON > 1 else 1.0


def matern_corr_np(r, smooth):
    r = np.asarray(r, dtype=np.float64)
    smooth = float(smooth)
    if smooth == 0.5:
        return np.exp(-r)
    z = np.sqrt(2.0 * smooth) * r
    safe = np.where(z < 1e-10, 1e-10, z)
    out = (2.0 ** (1.0 - smooth) / scipy_gamma(smooth)) * (safe ** smooth) * scipy_kv(smooth, safe)
    out = np.where(z < 1e-10, 1.0, out)
    out = np.nan_to_num(out, nan=0.0, posinf=1.0, neginf=0.0)
    return np.clip(out, 0.0, 1.0)


def simulate_spatial_matern_grid(rng, smooth, sigmasq, range_space, nugget):
    px, py = 2 * SIM_N_LAT, 2 * SIM_N_LON
    lag_lat = np.arange(px, dtype=np.float64) * SIM_LAT_STEP
    lag_lon = np.arange(py, dtype=np.float64) * SIM_LON_STEP
    lag_lat[px // 2:] -= px * SIM_LAT_STEP
    lag_lon[py // 2:] -= py * SIM_LON_STEP
    lag_lat_g, lag_lon_g = np.meshgrid(lag_lat, lag_lon, indexing='ij')
    dist = np.sqrt(lag_lat_g ** 2 + lag_lon_g ** 2) / float(range_space)
    cov = float(sigmasq) * matern_corr_np(dist, smooth)
    spec = np.fft.fftn(cov).real
    spec = np.clip(spec, 0.0, None)
    noise = np.fft.fftn(rng.standard_normal((px, py)))
    field = np.fft.ifftn(np.sqrt(spec) * noise).real[:SIM_N_LAT, :SIM_N_LON]

    lat_g, lon_g = np.meshgrid(sim_lat_vals, sim_lon_vals, indexing='ij')
    trend = (
        SIM_MEAN['intercept']
        + SIM_MEAN['lat_slope'] * (lat_g - float(np.mean(sim_lat_vals)))
        + SIM_MEAN['lon_slope'] * (lon_g - float(np.mean(sim_lon_vals)))
    )
    eps = rng.normal(0.0, np.sqrt(float(nugget)), size=field.shape)
    return trend + field + eps


rng = np.random.default_rng(SIM_SEED)
sim_hour_maps = {}
sim_full_values = {}
for hour_idx, key in enumerate(day_keys):
    hour_df_map = {key: df_map[key]}
    hour_map, _ = loader.load_working_data(
        hour_df_map,
        monthly_mean=monthly_mean,
        idx_for_datamap=[0, 1],
        ord_mm=grid_order,
        dtype=DTYPE,
        keep_ori=True,
    )
    base = next(iter(hour_map.values())).to(DEVICE).clone()
    sim_grid = simulate_spatial_matern_grid(
        rng,
        smooth=TRUE_SMOOTH,
        sigmasq=TRUE_PARAMS['sigmasq'],
        range_space=TRUE_PARAMS['range'],
        nugget=TRUE_PARAMS['nugget'],
    )
    full_y = sim_grid[sim_local_to_row, sim_local_to_col]
    obs_mask = torch.isfinite(base[:, 2]) & torch.isfinite(base[:, 0]) & torch.isfinite(base[:, 1])
    base[:, 2] = torch.nan
    base[obs_mask, 2] = torch.as_tensor(full_y[obs_mask.detach().cpu().numpy()], device=DEVICE, dtype=DTYPE)
    sim_hour_maps[key] = base.contiguous()
    sim_full_values[key] = full_y

print('simulation truth:', {'smooth': TRUE_SMOOTH, **TRUE_PARAMS})
print('simulation mean:', SIM_MEAN)
print('borrowed real day keys:', day_keys[0], '...', day_keys[-1])
print('full grid:', grid_coords_full.shape, 'regular grid:', (SIM_N_LAT, SIM_N_LON), 'step:', (SIM_LAT_STEP, SIM_LON_STEP))
print('valid obs by hour:', [int(torch.isfinite(sim_hour_maps[k][:, 2]).sum().item()) for k in day_keys])
print('2D resolution n_grid:', {resolution_label(s): int(np.sum((sim_local_to_row % int(s) == 0) & (sim_local_to_col % int(s) == 0))) for s in RESOLUTION_STRIDES})


In [ ]:
def load_hour_map(hour_idx):
    key = day_keys[int(hour_idx)]
    return {key: sim_hour_maps[key].clone().to(DEVICE)}, key


def thin_hour_map(hour_map, stride):
    # True 2D resolution thinning on the regular lat/lon grid:
    # x8 keeps every 8th latitude and every 8th longitude, not every 8th
    # element in the flattened vector. This makes the spectrum Nyquist cut
    # k_max / stride meaningful.
    stride = int(stride)
    keep = (sim_local_to_row % stride == 0) & (sim_local_to_col % stride == 0)
    thin_idx = np.flatnonzero(keep).astype(np.int64)
    thin_map = {k: v[thin_idx].contiguous() for k, v in hour_map.items()}
    thin_grid = np.ascontiguousarray(grid_coords_full[thin_idx])
    return thin_map, thin_grid, thin_idx


def build_model(variant, input_map, grid_coords, fit_smooth=SMOOTH):
    spec = VARIANTS[variant]
    return spec['class'](
        smooth=float(fit_smooth),
        input_map=input_map,
        grid_coords=np.ascontiguousarray(grid_coords.astype(np.float64)),
        block_shape=CLUSTER_SPEC['block_shape'],
        n_neighbor_blocks=CLUSTER_SPEC['n_neighbor_blocks'],
        target_chunk_size=CLUSTER_SPEC['target_chunk_size'],
        min_target_points=CLUSTER_SPEC['min_target_points'],
        mean_design=MEAN_DESIGN,
    )


def make_base_row(variant, hour_idx, time_key, stride, n_grid, n_valid, pre_s, fit_s, loss, fit_iter, est, diag, fit_smooth):
    spec = VARIANTS[variant]
    row = {
        'date_str': f'{YEAR}{MONTH:02d}{DAY_IDX + 1:02d}',
        'day_idx': DAY_IDX,
        'hour_idx': int(hour_idx),
        'time_key': str(time_key),
        'resolution_stride': int(stride),
        'resolution_label': resolution_label(stride),
        'variant': variant,
        'true_smooth': float(TRUE_SMOOTH),
        'fit_smooth': float(fit_smooth),
        'smooth': float(fit_smooth),
        'mean_design': MEAN_DESIGN,
        'fit_type': 'full',
        'model': spec['model'],
        'kernel': spec['kernel'],
        'coord_mode': 'regular-grid 2D every-k thinning, fixed thinned grid order; 4x4 cluster max-min ordering on cluster centroids; covariance on Source_Latitude/Source_Longitude',
        'loss': float(loss),
        'fit_iter_raw': int(fit_iter),
        'fit_steps_reported': int(fit_iter) + 1,
        'precompute_s': float(pre_s),
        'fit_s': float(fit_s),
        'total_s': float(pre_s + fit_s),
        'n_grid': int(n_grid),
        'n_valid': int(n_valid),
        'valid_fraction': float(n_valid / n_grid) if n_grid else np.nan,
        'est_sigmasq': float(est['sigmasq']),
        'est_range': float(est['range']),
        'est_nugget': float(est.get('nugget', 0.0)),
        **diag,
        'cluster_block_shape': '4x4',
        'cluster_neighbor_blocks': CLUSTER_SPEC['n_neighbor_blocks'],
        'total_conditioning_nominal': int(CLUSTER_SPEC['n_neighbor_blocks'] * np.prod(CLUSTER_SPEC['block_shape'])),
    }
    return row


In [ ]:
def fit_full_variant(variant, hour_idx, stride, fit_smooth):
    hour_map, time_key = load_hour_map(hour_idx)
    thin_map, thin_grid, _ = thin_hour_map(hour_map, stride)
    n_grid = int(thin_grid.shape[0])
    n_valid = count_valid(thin_map)
    print('\n' + '=' * 100)
    print(f'{variant} | true_smooth={TRUE_SMOOTH} | fit_smooth={fit_smooth} | hour={hour_idx + 1} | {time_key} | {resolution_label(stride)} | n_grid={n_grid:,} | n_valid={n_valid:,}')

    model = build_model(variant, thin_map, thin_grid, fit_smooth=fit_smooth)
    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre
    diag = space_diag(model)

    params = make_full_params(variant)
    opt = model.set_optimizer(params, lr=LBFGS_LR, max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL, history_size=LBFGS_HIST)
    t_fit = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, opt, max_steps=LBFGS_STEPS_FULL, grad_tol=GRAD_TOL)
    fit_s = time.time() - t_fit
    p_labels = VARIANTS[variant]['p_labels']
    est = backmap(out[:len(p_labels)], p_labels, variant)
    row = make_base_row(variant, hour_idx, time_key, stride, n_grid, n_valid, pre_s, fit_s, out[-1], fit_iter, est, diag, fit_smooth)
    print('RESULT:', {k: round(v, 4) if isinstance(v, float) else v for k, v in row.items() if k in ['variant','fit_smooth','resolution_label','loss','est_sigmasq','est_range','est_nugget','total_s']})

    del model, params, opt, hour_map, thin_map
    empty_cache()
    return row


In [ ]:
fit_rows = []
if RUN_FULL:
    for fit_smooth in FIT_SMOOTHS:
        for variant in VARIANTS:
            for hour_idx in HOUR_IDX_LIST:
                for stride in RESOLUTION_STRIDES:
                    fit_rows.append(fit_full_variant(variant, hour_idx, stride, fit_smooth))
                tmp = round_df(pd.DataFrame(fit_rows))
                tmp.to_csv(OUT_DIR / f'{OUT_PREFIX}_full.csv', index=False, float_format=f'%.{ROUND_DECIMALS}f')

fit_df = pd.DataFrame(fit_rows)
full_path = OUT_DIR / f'{OUT_PREFIX}_full.csv'
round_df(fit_df).to_csv(full_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved full fits:', full_path)
display(round_df(fit_df))

param_rows = []
for _, row in fit_df.iterrows():
    for p in ['sigmasq', 'range', 'nugget']:
        param_rows.append({
            'date_str': row['date_str'],
            'hour_idx': int(row['hour_idx']),
            'time_key': row['time_key'],
            'resolution_stride': int(row['resolution_stride']),
            'resolution_label': row['resolution_label'],
            'variant': row['variant'],
            'true_smooth': row.get('true_smooth', TRUE_SMOOTH),
            'fit_smooth': row.get('fit_smooth', row['smooth']),
            'smooth': row['smooth'],
            'parameter': p,
            'estimate': row[f'est_{p}'],
            'loss': row['loss'],
            'n_grid': row['n_grid'],
            'n_valid': row['n_valid'],
        })
param_df = pd.DataFrame(param_rows)
param_path = OUT_DIR / f'{OUT_PREFIX}_param_table.csv'
round_df(param_df).to_csv(param_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved param table:', param_path)


In [ ]:
# Full-fit results table.
plot_df = fit_df.copy()
plot_df['resolution_label'] = pd.Categorical(
    plot_df['resolution_label'], categories=[f'x{s}' for s in RESOLUTION_STRIDES], ordered=True
)
plot_df = plot_df.sort_values(['fit_smooth', 'variant', 'hour_idx', 'resolution_label'])

display(round_df(plot_df[['true_smooth','fit_smooth','variant','hour_idx','resolution_stride','resolution_label','loss','est_sigmasq','est_range','est_nugget','n_grid','n_valid','total_s']]))


In [ ]:
# %%
# Partial-profile fits anchored at the full-fit estimates.
# sigma_only: estimate sigmasq only; keep range and nugget fixed at the full-fit estimates.
# range_only: estimate range only; keep sigmasq and nugget fixed at the full-fit estimates.
RUN_PROFILE_FITS = True
PROFILE_TYPES = ['sigma_only', 'range_only']
PROFILE_LBFGS_STEPS = 8
PROFILE_GRAD_TOL = 1e-5
PROFILE_EPS = 1e-10


def safe_log_value(x, eps=PROFILE_EPS):
    return float(np.log(max(float(x), eps)))


def log_tensor_value(x, like, eps=PROFILE_EPS):
    return torch.as_tensor(np.log(max(float(x), eps)), device=like.device, dtype=like.dtype)


def profile_param_tensor(variant, profile_type, theta, anchor):
    if variant == 'nugget_free':
        vals = {
            'sigmasq': log_tensor_value(anchor['sigmasq'], theta),
            'range': log_tensor_value(anchor['range'], theta),
            'nugget': log_tensor_value(anchor['nugget'], theta),
        }
        vals['sigmasq' if profile_type == 'sigma_only' else 'range'] = theta.reshape(())
        return torch.stack([vals['sigmasq'], vals['range'], vals['nugget']])
    if variant == 'nugget0':
        vals = {
            'sigmasq': log_tensor_value(anchor['sigmasq'], theta),
            'range': log_tensor_value(anchor['range'], theta),
        }
        vals['sigmasq' if profile_type == 'sigma_only' else 'range'] = theta.reshape(())
        return torch.stack([vals['sigmasq'], vals['range']])
    raise ValueError(f'Unknown variant: {variant}')


def profile_estimate_from_theta(profile_type, theta, anchor):
    est = dict(anchor)
    est['sigmasq' if profile_type == 'sigma_only' else 'range'] = float(torch.exp(theta.detach()).cpu().item())
    return est


def optimize_profile(model, variant, profile_type, anchor):
    free_name = 'sigmasq' if profile_type == 'sigma_only' else 'range'
    theta = torch.tensor([safe_log_value(anchor[free_name])], device=DEVICE, dtype=DTYPE, requires_grad=True)
    optimizer = torch.optim.LBFGS(
        [theta],
        lr=LBFGS_LR,
        max_iter=LBFGS_EVAL,
        max_eval=LBFGS_EVAL,
        history_size=LBFGS_HIST,
        line_search_fn='strong_wolfe',
    )

    last_loss = None
    last_iter = 0
    for i in range(PROFILE_LBFGS_STEPS):
        last_iter = i

        def closure():
            optimizer.zero_grad()
            params = profile_param_tensor(variant, profile_type, theta, anchor)
            loss = model.vecchia_batched_likelihood(params)
            loss.backward()
            return loss

        last_loss = optimizer.step(closure)
        with torch.no_grad():
            max_grad = abs(float(theta.grad.detach().item())) if theta.grad is not None else 0.0
            print(f'    {profile_type} step {i + 1}/{PROFILE_LBFGS_STEPS}: loss={float(last_loss.detach().item()):.6f}, grad={max_grad:.2e}')
            if max_grad < PROFILE_GRAD_TOL:
                break

    est = profile_estimate_from_theta(profile_type, theta, anchor)
    loss = float(last_loss.detach().cpu().item()) if isinstance(last_loss, torch.Tensor) else np.nan
    return est, loss, last_iter


def fit_partial_profiles_for_row(row):
    variant = str(row['variant'])
    hour_idx = int(row['hour_idx'])
    stride = int(row['resolution_stride'])
    fit_smooth = float(row.get('fit_smooth', row.get('smooth', SMOOTH)))
    anchor = {
        'sigmasq': float(row['est_sigmasq']),
        'range': float(row['est_range']),
        'nugget': float(row.get('est_nugget', 0.0)),
    }

    hour_map, time_key = load_hour_map(hour_idx)
    thin_map, thin_grid, _ = thin_hour_map(hour_map, stride)
    model = build_model(variant, thin_map, thin_grid, fit_smooth=fit_smooth)
    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre
    diag = space_diag(model)

    out_rows = []
    for profile_type in PROFILE_TYPES:
        print('\n' + '-' * 90)
        print(f"profile={profile_type} | {variant} | true_smooth={TRUE_SMOOTH} | fit_smooth={fit_smooth} | hour={hour_idx + 1} | {resolution_label(stride)} | anchor={anchor}")
        t_fit = time.time()
        est, loss, fit_iter = optimize_profile(model, variant, profile_type, anchor)
        fit_s = time.time() - t_fit
        profile_row = dict(row)
        profile_row.update({
            'fit_type': profile_type,
            'profile_type': profile_type,
            'true_smooth': float(TRUE_SMOOTH),
            'fit_smooth': float(fit_smooth),
            'smooth': float(fit_smooth),
            'fixed_sigmasq': anchor['sigmasq'] if profile_type == 'range_only' else np.nan,
            'fixed_range': anchor['range'] if profile_type == 'sigma_only' else np.nan,
            'fixed_nugget': anchor['nugget'],
            'loss': float(loss),
            'fit_iter_raw': int(fit_iter),
            'fit_steps_reported': int(fit_iter) + 1,
            'precompute_s': float(pre_s),
            'fit_s': float(fit_s),
            'total_s': float(pre_s + fit_s),
            'est_sigmasq': float(est['sigmasq']),
            'est_range': float(est['range']),
            'est_nugget': float(est.get('nugget', 0.0)),
            **diag,
        })
        out_rows.append(profile_row)

    del model, hour_map, thin_map
    empty_cache()
    return out_rows


profile_path = OUT_DIR / f'{OUT_PREFIX}_partial_profile.csv'
if RUN_PROFILE_FITS:
    if 'fit_df' not in globals() or fit_df.empty:
        fit_df = pd.read_csv(OUT_DIR / f'{OUT_PREFIX}_full.csv')

    profile_rows = []
    for _, fit_row in fit_df.iterrows():
        profile_rows.extend(fit_partial_profiles_for_row(fit_row))
        tmp = round_df(pd.DataFrame(profile_rows))
        tmp.to_csv(profile_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')

    profile_df = pd.DataFrame(profile_rows)
    round_df(profile_df).to_csv(profile_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
else:
    profile_df = pd.read_csv(profile_path) if profile_path.exists() else pd.DataFrame()

print('Saved partial-profile fits:', profile_path)
display(round_df(profile_df[['true_smooth','fit_smooth','variant','profile_type','hour_idx','resolution_stride','resolution_label','loss','est_sigmasq','est_range','est_nugget','fixed_sigmasq','fixed_range','fixed_nugget','n_grid','n_valid','total_s']]))


# %%
# Combined parameter trajectories, one 3x3 matrix per fitted smooth.
if 'fit_df' not in globals() or fit_df.empty:
    fit_df = pd.read_csv(OUT_DIR / f'{OUT_PREFIX}_full.csv')
if 'profile_df' not in globals() or profile_df.empty:
    profile_df = pd.read_csv(OUT_DIR / f'{OUT_PREFIX}_partial_profile.csv')

full_plot_df = fit_df.copy()
prof_plot_df = profile_df.copy()
for df in [full_plot_df, prof_plot_df]:
    df['resolution_label'] = pd.Categorical(
        df['resolution_label'], categories=[f'x{s}' for s in RESOLUTION_STRIDES], ordered=True
    )

prof_anchor_variant = 'nugget0'

combined_param_values = {
    'sigmasq': [full_plot_df['est_sigmasq'], prof_plot_df.loc[prof_plot_df['profile_type'] == 'sigma_only', 'est_sigmasq']],
    'range': [full_plot_df['est_range'], prof_plot_df.loc[prof_plot_df['profile_type'] == 'range_only', 'est_range']],
    'nugget': [full_plot_df['est_nugget'], prof_plot_df['fixed_nugget']],
}
combined_param_ylims = {}
for p, series_list in combined_param_values.items():
    vals = pd.concat([pd.to_numeric(s, errors='coerce') for s in series_list], ignore_index=True)
    vals = vals.replace([np.inf, -np.inf], np.nan).dropna()
    if vals.empty:
        combined_param_ylims[p] = (0.0, 1.0)
        continue
    lo, hi = float(vals.min()), float(vals.max())
    pad = max(abs(hi) * 0.05, 0.05) if np.isclose(lo, hi) else 0.08 * (hi - lo)
    if p == 'nugget':
        lo = min(0.0, lo)
    combined_param_ylims[p] = (lo - pad, hi + pad)

params = [
    ('sigmasq', 'sigmasq'),
    ('range', 'range'),
    ('nugget', 'nugget'),
]
row_specs = [
    ('full_nugget0', 'full: nugget fixed 0'),
    ('full_nugget_free', 'full: nugget free'),
    ('profile_nugget0', 'profile anchor: nugget fixed 0'),
]

for fit_smooth in sorted(pd.to_numeric(full_plot_df['fit_smooth'], errors='coerce').dropna().unique()):
    full_s = full_plot_df[np.isclose(full_plot_df['fit_smooth'].astype(float), fit_smooth)].copy()
    prof_s = prof_plot_df[np.isclose(prof_plot_df['fit_smooth'].astype(float), fit_smooth)].copy()
    fig, axes = plt.subplots(3, 3, figsize=(16, 10.5), constrained_layout=True, sharex=True)
    for i, (row_key, row_title) in enumerate(row_specs):
        for j, (p, p_title) in enumerate(params):
            ax = axes[i, j]
            if row_key == 'full_nugget0':
                sub = full_s[full_s['variant'] == 'nugget0'].sort_values(['hour_idx', 'resolution_label'])
                y_col = f'est_{p}'
            elif row_key == 'full_nugget_free':
                sub = full_s[full_s['variant'] == 'nugget_free'].sort_values(['hour_idx', 'resolution_label'])
                y_col = f'est_{p}'
            else:
                if p == 'sigmasq':
                    sub = prof_s[(prof_s['variant'] == prof_anchor_variant) & (prof_s['profile_type'] == 'sigma_only')].sort_values(['hour_idx', 'resolution_label'])
                    y_col = 'est_sigmasq'
                elif p == 'range':
                    sub = prof_s[(prof_s['variant'] == prof_anchor_variant) & (prof_s['profile_type'] == 'range_only')].sort_values(['hour_idx', 'resolution_label'])
                    y_col = 'est_range'
                else:
                    sub = prof_s[(prof_s['variant'] == prof_anchor_variant) & (prof_s['profile_type'] == 'sigma_only')].sort_values(['hour_idx', 'resolution_label'])
                    y_col = 'fixed_nugget'
            if sub.empty or y_col not in sub:
                ax.set_visible(False)
                continue
            for hour_idx, hs in sub.groupby('hour_idx'):
                ax.plot(hs['resolution_label'].astype(str), hs[y_col], marker='o', linewidth=1.5, label=f'h{int(hour_idx) + 1}')
            ax.set_title(f'{row_title} | {p_title}')
            ax.set_xlabel('resolution thinning')
            ax.set_ylabel('estimate')
            ax.set_ylim(*combined_param_ylims[p])
            ax.grid(True, alpha=0.25)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='center right', bbox_to_anchor=(1.01, 0.5), fontsize=8, title='hour')
    fig.suptitle(
        f'Pure-space isotropic clusterB2 Vecchia simulation: true smooth={TRUE_SMOOTH}, fitted smooth={fit_smooth}',
        fontsize=14,
    )
    param_plot_path = OUT_DIR / f'{OUT_PREFIX}_param_trajectories_fit_s{smooth_tag(fit_smooth)}.png'
    fig.savefig(param_plot_path, dpi=180, bbox_inches='tight')
    print('Saved parameter plot:', param_plot_path)
    plt.show()


## Spectral diagnostic

The empirical spectrum uses the same thinned points as each fit. Residuals are computed after an OLS `intercept + lat + lon` detrend, placed back onto the full grid, tapered, zero-filled, FFTed, and radially averaged. For `nugget_free`, the theoretical spectrum includes the fitted flat nugget component before being scaled to the empirical curve.


In [ ]:
RADIAL_N_BINS = 70
RADIAL_QMAX = 0.985
APPLY_HANN = True
EPS = 1e-12

_lat_key = np.round(grid_coords_full[:, 0], 10)
_lon_key = np.round(grid_coords_full[:, 1], 10)
lat_vals = np.sort(np.unique(_lat_key))
lon_vals = np.sort(np.unique(_lon_key))
lat_to_row = {float(v): i for i, v in enumerate(lat_vals)}
lon_to_col = {float(v): i for i, v in enumerate(lon_vals)}
local_to_row = np.asarray([lat_to_row[float(v)] for v in _lat_key], dtype=np.int64)
local_to_col = np.asarray([lon_to_col[float(v)] for v in _lon_key], dtype=np.int64)
N_LAT, N_LON = len(lat_vals), len(lon_vals)
LAT_STEP = float(np.median(np.diff(lat_vals))) if N_LAT > 1 else 1.0
LON_STEP = float(np.median(np.diff(lon_vals))) if N_LON > 1 else 1.0
print('spectral full grid:', (N_LAT, N_LON), 'lat/lon step:', LAT_STEP, LON_STEP)


def detrended_residual_grid(hour_idx, stride):
    hour_map, time_key = load_hour_map(hour_idx)
    thin_map, thin_grid, thin_idx = thin_hour_map(hour_map, stride)
    arr = next(iter(thin_map.values())).detach().cpu().numpy()
    y = arr[:, 2].astype(float)
    lat = arr[:, 0].astype(float)
    lon = arr[:, 1].astype(float)
    valid = np.isfinite(y) & np.isfinite(lat) & np.isfinite(lon)
    if valid.sum() < 4:
        raise ValueError(f'Not enough valid points for hour={hour_idx}, stride={stride}')

    X = np.column_stack([
        np.ones(valid.sum()),
        lat[valid] - np.nanmean(lat[valid]),
        lon[valid] - np.nanmean(lon[valid]),
    ])
    beta, *_ = np.linalg.lstsq(X, y[valid], rcond=None)
    X_all = np.column_stack([
        np.ones(len(y)),
        lat - np.nanmean(lat[valid]),
        lon - np.nanmean(lon[valid]),
    ])
    resid = y - X_all @ beta

    grid = np.full((N_LAT, N_LON), np.nan, dtype=float)
    mask = np.zeros((N_LAT, N_LON), dtype=float)
    keep_idx = thin_idx[valid]
    rr = local_to_row[keep_idx]
    cc = local_to_col[keep_idx]
    grid[rr, cc] = resid[valid]
    mask[rr, cc] = 1.0
    return grid, mask, time_key, int(valid.sum())


def masked_periodogram(grid, mask):
    z = np.nan_to_num(grid, nan=0.0).astype(float)
    z = z - (np.sum(z * mask) / max(float(mask.sum()), 1.0))
    win = np.outer(np.hanning(z.shape[0]), np.hanning(z.shape[1])) if APPLY_HANN else np.ones_like(z)
    zw = z * win
    norm = np.sum((mask * win) ** 2)
    norm = norm if norm > EPS else 1.0
    return np.abs(np.fft.fftshift(np.fft.fft2(zw))) ** 2 / norm


def frequency_grid():
    fy = np.fft.fftshift(np.fft.fftfreq(N_LAT, d=LAT_STEP))
    fx = np.fft.fftshift(np.fft.fftfreq(N_LON, d=LON_STEP))
    omega_y = 2.0 * np.pi * fy
    omega_x = 2.0 * np.pi * fx
    OX, OY = np.meshgrid(omega_x, omega_y)
    k = np.sqrt(OX ** 2 + OY ** 2)
    return k, OX ** 2 + OY ** 2


K_RADIAL, OMEGA2 = frequency_grid()
_positive_k = K_RADIAL[np.isfinite(K_RADIAL) & (K_RADIAL > 0)]
K_MAX = float(np.quantile(_positive_k, RADIAL_QMAX))
RADIAL_BINS = np.linspace(0.0, K_MAX, RADIAL_N_BINS + 1)


def matern_spectrum_shape(sigmasq, range_, nugget, smooth):
    nu = float(smooth)
    alpha = 2.0 * nu / max(float(range_) ** 2, EPS)
    matern = float(sigmasq) * (alpha + OMEGA2) ** (-(nu + 1.0))
    return matern + max(float(nugget), 0.0)


def radial_average(surface):
    vals = np.asarray(surface, dtype=float).ravel()
    kk = K_RADIAL.ravel()
    good = np.isfinite(vals) & np.isfinite(kk) & (kk > 0) & (kk <= K_MAX)
    bin_idx = np.digitize(kk[good], RADIAL_BINS) - 1
    rows = []
    vg = vals[good]
    for b in range(RADIAL_N_BINS):
        m = bin_idx == b
        if not np.any(m):
            continue
        rows.append({
            'k_bin': b,
            'k_mid': 0.5 * (RADIAL_BINS[b] + RADIAL_BINS[b + 1]),
            'spectrum': float(np.nanmean(vg[m])),
            'n_freq': int(m.sum()),
        })
    return pd.DataFrame(rows)


In [ ]:
if RUN_SPECTRUM:
    full_path = OUT_DIR / f'{OUT_PREFIX}_full.csv'
    if full_path.exists():
        fit_df = pd.read_csv(full_path)
    elif 'fit_df' not in globals() or fit_df.empty:
        raise FileNotFoundError(f'Full fit CSV not found: {full_path}')

    fit_df = fit_df.copy()
    if 'fit_smooth' not in fit_df.columns:
        fit_df['fit_smooth'] = fit_df['smooth']
    fit_df['fit_smooth'] = pd.to_numeric(fit_df['fit_smooth'], errors='coerce')
    print('Full-fit rows by fitted smooth:')
    display(fit_df.groupby(['fit_smooth', 'variant'], dropna=False).size().rename('n').reset_index())

    spectral_rows = []
    for r in fit_df.itertuples(index=False):
        fit_smooth = float(r.fit_smooth)
        grid, mask, time_key, n_valid_spectrum = detrended_residual_grid(int(r.hour_idx), int(r.resolution_stride))
        data_p = masked_periodogram(grid, mask)
        theory_p = matern_spectrum_shape(r.est_sigmasq, r.est_range, r.est_nugget, fit_smooth)

        data_rad = radial_average(data_p).rename(columns={'spectrum': 'data_spectrum'})
        theory_rad = radial_average(theory_p).rename(columns={'spectrum': 'theory_spectrum'})
        merged = data_rad.merge(theory_rad[['k_bin', 'theory_spectrum']], on='k_bin', how='inner')

        d = merged['data_spectrum'].to_numpy(dtype=float)
        t = merged['theory_spectrum'].to_numpy(dtype=float)
        good = np.isfinite(d) & np.isfinite(t) & (d > 0) & (t > 0)
        scale = float(np.nanmedian(d[good] / t[good])) if good.any() else 1.0
        merged['theory_spectrum_scaled'] = merged['theory_spectrum'] * scale

        for m in merged.itertuples(index=False):
            spectral_rows.append({
                'date_str': r.date_str,
                'true_smooth': float(getattr(r, 'true_smooth', TRUE_SMOOTH)),
                'fit_smooth': float(fit_smooth),
                'hour_idx': int(r.hour_idx),
                'time_key': time_key,
                'resolution_stride': int(r.resolution_stride),
                'resolution_label': r.resolution_label,
                'variant': r.variant,
                'smooth': float(fit_smooth),
                'n_valid_spectrum': n_valid_spectrum,
                'est_sigmasq': float(r.est_sigmasq),
                'est_range': float(r.est_range),
                'est_nugget': float(r.est_nugget),
                'k_bin': int(m.k_bin),
                'k_mid': float(m.k_mid),
                'n_freq': int(m.n_freq),
                'data_spectrum': float(m.data_spectrum),
                'theory_spectrum_raw': float(m.theory_spectrum),
                'theory_spectrum_scaled': float(m.theory_spectrum_scaled),
                'theory_scale_to_data': scale,
            })

    spectral_df = pd.DataFrame(spectral_rows)
    spectral_path = OUT_DIR / f'{OUT_PREFIX}_radial_spectrum.csv'
    spectral_df.to_csv(spectral_path, index=False)
    print('Saved radial spectrum:', spectral_path)
    print('Spectrum rows by fitted smooth:')
    display(spectral_df.groupby(['fit_smooth', 'variant'], dropna=False).size().rename('n').reset_index())
    missing = sorted(set(np.round(FIT_SMOOTHS, 8)) - set(np.round(spectral_df['fit_smooth'].dropna().unique(), 8)))
    if missing:
        print('WARNING: missing fitted smooths in spectral_df:', missing)
    display(round_df(
        spectral_df
        .sort_values(['fit_smooth', 'variant', 'hour_idx', 'resolution_stride', 'k_bin'])
        .groupby(['fit_smooth', 'variant'], group_keys=False)
        .head(6)
    ))


In [ ]:
# %%
# Resolution-aware spectrum summaries.
# Data spectrum is retained only below each thinning's effective Nyquist.
# Fitted theoretical spectrum is retained on the full-grid frequency axis.
spectral_path = OUT_DIR / f'{OUT_PREFIX}_radial_spectrum.csv'
spectral_df = pd.read_csv(spectral_path)
spectral_df['fit_smooth'] = pd.to_numeric(spectral_df['fit_smooth'], errors='coerce')
print('Loaded spectral fit_smooth values:', sorted(spectral_df['fit_smooth'].dropna().unique()))

plot_spec = spectral_df.copy()
plot_spec['resolution_label'] = pd.Categorical(
    plot_spec['resolution_label'], categories=[f'x{s}' for s in RESOLUTION_STRIDES], ordered=True
)

FULL_K_PLOT_MAX = float(plot_spec['k_mid'].max())
plot_spec['effective_k_max'] = FULL_K_PLOT_MAX / plot_spec['resolution_stride'].astype(float)
plot_spec_low = plot_spec[plot_spec['k_mid'] <= plot_spec['effective_k_max']].copy()

avg_low_data = (
    plot_spec_low
    .groupby(['fit_smooth', 'variant', 'resolution_label', 'resolution_stride', 'k_bin'], observed=False)
    .agg(
        k_mid=('k_mid', 'mean'),
        data_spectrum=('data_spectrum', 'mean'),
        n_hours=('hour_idx', 'nunique'),
        effective_k_max=('effective_k_max', 'mean'),
    )
    .reset_index()
)
low_path = OUT_DIR / f'{OUT_PREFIX}_radial_spectrum_lowfreq_only_avg.csv'
avg_low_data.to_csv(low_path, index=False)
print('Saved low-frequency data spectrum:', low_path)

avg_theory_full = (
    plot_spec
    .groupby(['fit_smooth', 'variant', 'resolution_label', 'k_bin'], observed=False)
    .agg(
        k_mid=('k_mid', 'mean'),
        theory_spectrum_scaled=('theory_spectrum_scaled', 'mean'),
    )
    .reset_index()
)


In [ ]:
# %%
# Spectral diagnostics for partial-profile fits, one 3x4 matrix per fitted smooth.
profile_df = pd.read_csv(OUT_DIR / f'{OUT_PREFIX}_partial_profile.csv')
profile_df['fit_smooth'] = pd.to_numeric(profile_df['fit_smooth'], errors='coerce')
print('Partial-profile rows by fitted smooth:')
display(profile_df.groupby(['fit_smooth', 'variant', 'profile_type'], dropna=False).size().rename('n').reset_index())

if RUN_SPECTRUM and not profile_df.empty:
    profile_spectral_rows = []
    for r in profile_df.itertuples(index=False):
        fit_smooth = float(getattr(r, 'fit_smooth', getattr(r, 'smooth')))
        grid, mask, time_key, n_valid_spectrum = detrended_residual_grid(int(r.hour_idx), int(r.resolution_stride))
        data_p = masked_periodogram(grid, mask)
        theory_p = matern_spectrum_shape(r.est_sigmasq, r.est_range, r.est_nugget, fit_smooth)

        data_rad = radial_average(data_p).rename(columns={'spectrum': 'data_spectrum'})
        theory_rad = radial_average(theory_p).rename(columns={'spectrum': 'theory_spectrum'})
        merged = data_rad.merge(theory_rad[['k_bin', 'theory_spectrum']], on='k_bin', how='inner')

        d = merged['data_spectrum'].to_numpy(dtype=float)
        t = merged['theory_spectrum'].to_numpy(dtype=float)
        good = np.isfinite(d) & np.isfinite(t) & (d > 0) & (t > 0)
        scale = float(np.exp(np.mean(np.log(d[good]) - np.log(t[good])))) if good.any() else 1.0
        merged['theory_spectrum_scaled'] = merged['theory_spectrum'] * scale

        for m in merged.itertuples(index=False):
            profile_spectral_rows.append({
                'date_str': str(r.date_str),
                'true_smooth': float(getattr(r, 'true_smooth', TRUE_SMOOTH)),
                'fit_smooth': float(fit_smooth),
                'hour_idx': int(r.hour_idx),
                'time_key': str(r.time_key),
                'resolution_stride': int(r.resolution_stride),
                'resolution_label': str(r.resolution_label),
                'variant': str(r.variant),
                'profile_type': str(r.profile_type),
                'smooth': float(fit_smooth),
                'n_valid_spectrum': int(n_valid_spectrum),
                'est_sigmasq': float(r.est_sigmasq),
                'est_range': float(r.est_range),
                'est_nugget': float(r.est_nugget),
                'k_bin': int(m.k_bin),
                'k_mid': float(m.k_mid),
                'n_freq': int(m.n_freq),
                'data_spectrum': float(m.data_spectrum),
                'theory_spectrum_raw': float(m.theory_spectrum),
                'theory_spectrum_scaled': float(m.theory_spectrum_scaled),
                'theory_scale_to_data': scale,
            })

    profile_spectral_df = pd.DataFrame(profile_spectral_rows)
    profile_spectral_path = OUT_DIR / f'{OUT_PREFIX}_partial_profile_radial_spectrum.csv'
    profile_spectral_df.to_csv(profile_spectral_path, index=False)
    print('Saved partial-profile radial spectrum:', profile_spectral_path)

    plot_prof = profile_spectral_df.copy()
    plot_prof['resolution_label'] = pd.Categorical(
        plot_prof['resolution_label'], categories=[f'x{s}' for s in RESOLUTION_STRIDES], ordered=True
    )
    if 'FULL_K_PLOT_MAX' not in globals():
        FULL_K_PLOT_MAX = float(plot_prof['k_mid'].max())
    plot_prof['effective_k_max'] = FULL_K_PLOT_MAX / plot_prof['resolution_stride'].astype(float)
    plot_prof_low = plot_prof[plot_prof['k_mid'] <= plot_prof['effective_k_max']].copy()

    avg_prof_low = (
        plot_prof_low
        .groupby(['fit_smooth', 'variant', 'profile_type', 'resolution_label', 'resolution_stride', 'k_bin'], observed=False)
        .agg(
            k_mid=('k_mid', 'mean'),
            data_spectrum=('data_spectrum', 'mean'),
            n_hours=('hour_idx', 'nunique'),
            effective_k_max=('effective_k_max', 'mean'),
        )
        .reset_index()
    )
    avg_prof_theory = (
        plot_prof
        .groupby(['fit_smooth', 'variant', 'profile_type', 'resolution_label', 'k_bin'], observed=False)
        .agg(
            k_mid=('k_mid', 'mean'),
            theory_spectrum_scaled=('theory_spectrum_scaled', 'mean'),
        )
        .reset_index()
    )
    avg_prof_low_path = OUT_DIR / f'{OUT_PREFIX}_partial_profile_radial_spectrum_lowfreq_only_avg.csv'
    avg_prof_low.to_csv(avg_prof_low_path, index=False)
    print('Saved partial-profile low-frequency data spectrum:', avg_prof_low_path)

    positive_spectrum_vals = pd.concat(
        [
            pd.to_numeric(avg_low_data['data_spectrum'], errors='coerce'),
            pd.to_numeric(avg_theory_full['theory_spectrum_scaled'], errors='coerce'),
            pd.to_numeric(avg_prof_low['data_spectrum'], errors='coerce'),
            pd.to_numeric(avg_prof_theory['theory_spectrum_scaled'], errors='coerce'),
        ],
        ignore_index=True,
    ).replace([np.inf, -np.inf], np.nan).dropna()
    positive_spectrum_vals = positive_spectrum_vals[positive_spectrum_vals > 0]
    if positive_spectrum_vals.empty:
        SPECTRUM_YLIM = (1e-1, 1e4)
    else:
        SPECTRUM_YLIM = (
            10 ** np.floor(np.log10(float(positive_spectrum_vals.min()))),
            10 ** np.ceil(np.log10(float(positive_spectrum_vals.max()))),
        )
    print('Fixed combined spectrum y-limits:', SPECTRUM_YLIM)

    labels_order = [f'x{s}' for s in RESOLUTION_STRIDES]
    row_specs = [
        ('nugget0', 'full: nugget fixed 0'),
        ('nugget_free', 'full: nugget free'),
        ('profiles', 'nugget0 partial profiles'),
    ]

    fit_smooth_values = sorted(set(pd.to_numeric(plot_spec['fit_smooth'], errors='coerce').dropna().unique()).union(set(pd.to_numeric(plot_prof['fit_smooth'], errors='coerce').dropna().unique())))
    print('Plotting spectrum matrices for fitted smooths:', fit_smooth_values)
    for fit_smooth in fit_smooth_values:
        fig, axes = plt.subplots(len(row_specs), len(labels_order), figsize=(18, 10.5), constrained_layout=True, sharex=True, sharey=True)
        for i, (row_key, row_title) in enumerate(row_specs):
            for j, label in enumerate(labels_order):
                ax = axes[i, j]
                if row_key == 'profiles':
                    sub_data = avg_prof_low[
                        np.isclose(avg_prof_low['fit_smooth'].astype(float), fit_smooth)
                        & (avg_prof_low['variant'] == 'nugget0')
                        & (avg_prof_low['profile_type'] == 'sigma_only')
                        & (avg_prof_low['resolution_label'].astype(str) == label)
                        & (avg_prof_low['k_mid'] > 0)
                    ].copy()
                    sub_sigma = avg_prof_theory[
                        np.isclose(avg_prof_theory['fit_smooth'].astype(float), fit_smooth)
                        & (avg_prof_theory['variant'] == 'nugget0')
                        & (avg_prof_theory['profile_type'] == 'sigma_only')
                        & (avg_prof_theory['resolution_label'].astype(str) == label)
                        & (avg_prof_theory['k_mid'] > 0)
                    ].copy()
                    sub_range = avg_prof_theory[
                        np.isclose(avg_prof_theory['fit_smooth'].astype(float), fit_smooth)
                        & (avg_prof_theory['variant'] == 'nugget0')
                        & (avg_prof_theory['profile_type'] == 'range_only')
                        & (avg_prof_theory['resolution_label'].astype(str) == label)
                        & (avg_prof_theory['k_mid'] > 0)
                    ].copy()
                    hour_sub = plot_prof_low[
                        np.isclose(plot_prof_low['fit_smooth'].astype(float), fit_smooth)
                        & (plot_prof_low['variant'] == 'nugget0')
                        & (plot_prof_low['profile_type'] == 'sigma_only')
                        & (plot_prof_low['resolution_label'].astype(str) == label)
                        & (plot_prof_low['k_mid'] > 0)
                    ]
                    if sub_data.empty or sub_sigma.empty or sub_range.empty:
                        ax.set_visible(False)
                        continue
                    k_cut = float(sub_data['effective_k_max'].iloc[0])
                    for _, hs in hour_sub.groupby('hour_idx'):
                        ax.plot(hs['k_mid'], hs['data_spectrum'], color='0.7', alpha=0.20, linewidth=0.7)
                    ax.plot(sub_data['k_mid'], sub_data['data_spectrum'], color='black', linewidth=2.0, label='data residual spectrum')
                    ax.plot(sub_sigma['k_mid'], sub_sigma['theory_spectrum_scaled'], color='tab:blue', linewidth=1.8, linestyle='--', label='sigma-only theory')
                    ax.plot(sub_range['k_mid'], sub_range['theory_spectrum_scaled'], color='tab:green', linewidth=1.8, linestyle='--', label='range-only theory')
                else:
                    sub_data = avg_low_data[
                        np.isclose(avg_low_data['fit_smooth'].astype(float), fit_smooth)
                        & (avg_low_data['variant'] == row_key)
                        & (avg_low_data['resolution_label'].astype(str) == label)
                        & (avg_low_data['k_mid'] > 0)
                    ].copy()
                    sub_theory = avg_theory_full[
                        np.isclose(avg_theory_full['fit_smooth'].astype(float), fit_smooth)
                        & (avg_theory_full['variant'] == row_key)
                        & (avg_theory_full['resolution_label'].astype(str) == label)
                        & (avg_theory_full['k_mid'] > 0)
                    ].copy()
                    hour_sub = plot_spec_low[
                        np.isclose(plot_spec_low['fit_smooth'].astype(float), fit_smooth)
                        & (plot_spec_low['variant'] == row_key)
                        & (plot_spec_low['resolution_label'].astype(str) == label)
                        & (plot_spec_low['k_mid'] > 0)
                    ]
                    if sub_data.empty or sub_theory.empty:
                        ax.set_visible(False)
                        continue
                    k_cut = float(sub_data['effective_k_max'].iloc[0])
                    for _, hs in hour_sub.groupby('hour_idx'):
                        ax.plot(hs['k_mid'], hs['data_spectrum'], color='0.7', alpha=0.20, linewidth=0.7)
                    ax.plot(sub_data['k_mid'], sub_data['data_spectrum'], color='black', linewidth=2.0, label='data residual spectrum')
                    ax.plot(sub_theory['k_mid'], sub_theory['theory_spectrum_scaled'], color='tab:red', linewidth=1.8, linestyle='--', label='fitted theory')
                ax.axvline(k_cut, color='0.75', linestyle=':', linewidth=1.0)
                ax.set_xlim(0, FULL_K_PLOT_MAX)
                ax.set_ylim(*SPECTRUM_YLIM)
                ax.set_yscale('log')
                ax.set_title(f'{row_title}, {label} (data k <= {k_cut:.1f})')
                ax.set_xlabel('radial wavenumber on full-grid scale')
                if j == 0:
                    ax.set_ylabel('spectrum')
                ax.grid(True, alpha=0.25)
        axes[0, 0].legend(fontsize=8)
        fig.suptitle(f'true smooth={TRUE_SMOOTH}, fitted smooth={fit_smooth}: full-fit and partial-profile spectra', fontsize=14)
        spectrum_plot_path = OUT_DIR / f'{OUT_PREFIX}_combined_full_profile_spectrum_fit_s{smooth_tag(fit_smooth)}.png'
        fig.savefig(spectrum_plot_path, dpi=180, bbox_inches='tight')
        print('Saved combined spectrum plot:', spectrum_plot_path)
        plt.show()
